In [ ]:
!pip install pandas transformers datasets evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd

# --- 1. Load Datasets ---
url1 = "https://raw.githubusercontent.com/singhnivedita/SemEval2020-Task9/master/Fully%20Processed%20Datasets/FinalTrainingOnly.tsv"
url2 = "https://raw.githubusercontent.com/saanvibehele/hinglish-sentiment-analysis/main/data-main/data.csv"

try:
    df1 = pd.read_csv(url1, sep='\t')
    df2 = pd.read_csv(url2)

    # --- 2. IMPORTANT: Inspect Columns ---
    print("--- Inspect Dataset 1 (df1) ---")
    print("Columns found:", df1.columns)
    print(df1.head())

    print("\n--- Inspect Dataset 2 (df2) ---")
    print("Columns found:", df2.columns)
    print(df2.head())

except Exception as e:
    print(f"An error occurred loading the data: {e}")


--- Inspect Dataset 1 (df1) ---
Columns found: Index(['3',
       'pakistan ka ghra tauq pakistan israel ko tasleem nahein kerta isko palestin kehta - occupi palestin nan',
       '0'],
      dtype='object')
    3  \
0  41   
1  48   
2  64   
3  66   
4  68   

  pakistan ka ghra tauq pakistan israel ko tasleem nahein kerta isko palestin kehta - occupi palestin nan  \
0  madarchod mull ye mathura nahi dikha tha jab m...                                                        
1  manya pradhan mantri mahoday shriman narendra ...                                                        
2            krishna jcb full trend chal rahi aa nan                                                        
3  loksabha janta sirf modi ko vote de rahi thi n...                                                        
4       bhosdik tum pechvad ki tatti hi rahog bc nan                                                        

   0  
0  0  
1  2  
2  2  
3  2  
4  0  

--- Inspect Dataset 2 (df2) ---
Columns

In [ ]:
import pandas as pd
from datasets import Dataset

# --- 1. Load Datasets ---
url1 = "https://raw.githubusercontent.com/singhnivedita/SemEval2020-Task9/master/Fully%20Processed%20Datasets/FinalTrainingOnly.tsv"
url2 = "https://raw.githubusercontent.com/saanvibehele/hinglish-sentiment-analysis/main/data-main/data.csv"

try:
    # --- FIX for df1 ---
    # Load df1 WITHOUT a header, since it doesn't have one.
    # We will refer to columns by their number (index)
    df1 = pd.read_csv(url1, sep='\t', header=None)
    print("--- Loaded df1 (with header=None) ---")

    # Load df2 normally and clean its column names
    df2 = pd.read_csv(url2)
    df2.columns = df2.columns.str.strip() # Clean spaces
    print("--- Loaded df2 (and cleaned column names) ---")

    # --- 2. Define Your Column Names/Indices ---

    # For df1, we use column *indices* (numbers) based on your output:
    # Column 1 = text
    # Column 2 = label
    TEXT_COL_DF1_IDX = 1  # The 2nd column
    LABEL_COL_DF1_IDX = 2 # The 3rd column

    # For df2, we use column *names* based on your output:
    TEXT_COL_DF2 = 'Tweet'
    LABEL_COL_DF2 = 'Sentiment Polarity'

    # --- 3. Clean and Standardize Labels ---

    # Process Dataset 1 using its indices
    # We select columns by their number, then rename them to 'text' and 'label'
    df1_clean = df1.iloc[:, [TEXT_COL_DF1_IDX, LABEL_COL_DF1_IDX]].copy()
    df1_clean.columns = ['text', 'label']

    # Process Dataset 2 using its column names
    df2_clean = df2[[TEXT_COL_DF2, LABEL_COL_DF2]].copy()
    df2_clean.rename(columns={TEXT_COL_DF2: 'text', LABEL_COL_DF2: 'label'}, inplace=True)

    # --- 4. Standardize all labels to text ---

    # Map df1 labels (0, 1, 2) to text
    if df1_clean['label'].dtype in ['int64', 'float64']:
        print("Mapping df1 numeric labels (0, 1, 2) to text...")
        label_map_1 = {0: 'negative', 1: 'neutral', 2: 'positive'}
        df1_clean['label'] = df1_clean['label'].map(label_map_1)

    # Map df2 labels ('positive', 'negative', 'neutral')
    # We just need to make sure they are lowercase
    df2_clean['label'] = df2_clean['label'].str.lower().str.strip()


    # --- 5. Merge Datasets ---
    df_merged = pd.concat([df1_clean, df2_clean], ignore_index=True)

    # --- 6. Final Check ---
    print("\n--- Merged Dataset Info ---")
    df_merged.dropna(inplace=True) # Remove any rows with missing text or labels
    print(f"Total entries after cleaning: {len(df_merged)}")
    print("Label distribution in new dataset:")
    print(df_merged['label'].value_counts())

    # --- 7. Convert to HuggingFace Dataset Object ---
    # This creates the 'dataset' variable
    dataset = Dataset.from_pandas(df_merged)

    print("\nSuccessfully converted to HuggingFace Dataset:")
    print(dataset)

except Exception as e:
    print(f"\nAN ERROR OCCURRED: {e}")
    print("Please check the code and your data.")


--- Loaded df1 (with header=None) ---
--- Loaded df2 (and cleaned column names) ---
Mapping df1 numeric labels (0, 1, 2) to text...

--- Merged Dataset Info ---
Total entries after cleaning: 31560
Label distribution in new dataset:
label
neutral     11809
positive    10499
negative     9252
Name: count, dtype: int64

Successfully converted to HuggingFace Dataset:
Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 31560
})


In [ ]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer
import numpy as np

# --- 1. Define Model and Tokenizer ---
model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# --- 2. Create Label Mappings ---
labels_dict = {'negative': 0, 'neutral': 1, 'positive': 2}
num_labels = len(labels_dict)

# --- 3. Create Label Mapping Function (FIXED FOR BATCHES) ---
# When batched=True, 'example_batch['label']' is a list of labels.
# We must process each item *in* that list.
def map_labels(example_batch):
    example_batch['label'] = [labels_dict.get(lbl, -1) for lbl in example_batch['label']]
    return example_batch

# --- 4. Create Tokenizing Function ---
def preprocess_function(examples):
    clean_text = [str(t) for t in examples['text']]
    return tokenizer(clean_text, truncation=True, padding='max_length', max_length=128)

# --- 5. Apply Everything ---

print("Starting to split dataset...")
dataset_split = dataset.train_test_split(test_size=0.2)
print("Dataset split complete.")

print("Starting label mapping...")
# The map_labels function now correctly handles the batch
dataset_mapped = dataset_split.map(map_labels, batched=True).filter(
    lambda example: example['label'] != -1
)
print("Label mapping complete.")

print("Starting tokenization...")
tokenized_dataset = dataset_mapped.map(preprocess_function, batched=True)
print("Tokenization complete.")


print("\n--- Tokenized Dataset ---")
print(tokenized_dataset['train'][0])
print(f"\nSuccessfully created train/test split. Ready for training.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

Filter:   0%|          | 0/6312 [00:00<?, ? examples/s]

Label mapping complete.
Starting tokenization...


Map:   0%|          | 0/25248 [00:00<?, ? examples/s]

Map:   0%|          | 0/6312 [00:00<?, ? examples/s]

Tokenization complete.

--- Tokenized Dataset ---
{'text': 'waah gaze case opposit liter kick life cos clingi friend lost nan', 'label': 1, '__index_level_0__': 14330, 'input_ids': [0, 259, 1366, 53462, 7225, 233, 77087, 15659, 101630, 6897, 9545, 33139, 22242, 34391, 72856, 25990, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# --- 1. Load the Model ---
print("Loading pre-trained model...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels
)
print("Model loaded.")

# --- 2. Define Training Arguments (BARE MINIMUM) ---
print("Using BARE MINIMUM TrainingArguments.")
training_args = TrainingArguments(
    output_dir="hinglish_sentiment_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,

    # --- THIS IS THE FIX ---
    # This tells the Trainer to NOT log to wandb
    report_to="none",
)

# --- 3. Initialize the Trainer (BARE MINIMUM) ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],    # From your Step 3
    tokenizer=tokenizer,
)

# --- 4. START TRAINING ---
print("\n--- STARTING MODEL TRAINING (Simplified) ---")
trainer.train()
print("--- TRAINING COMPLETE ---")

# --- 5. Save the Final Model ---
trainer.save_model("my_final_hinglish_model")
tokenizer.save_pretrained("my_final_hinglish_model")
print("Final model saved to 'my_final_hinglish_model'")

Loading pre-trained model...


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-571004637.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Model loaded.
Using BARE MINIMUM TrainingArguments.

--- STARTING MODEL TRAINING (Simplified) ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
from transformers import pipeline

# --- 1. Load your saved model into a 'pipeline' ---
# This is the easiest way to get predictions.
print("Loading model from 'my_final_hinglish_model'...")
sentiment_pipeline = pipeline(
    "text-classification",
    model="my_final_hinglish_model",
    tokenizer="my_final_hinglish_model"
)
print("Model loaded successfully.")

# --- 2. Create a lookup for the labels ---
# The model will output 'LABEL_0', 'LABEL_1', 'LABEL_2'
# We need to map these back to words.
labels_dict = {'negative': 0, 'neutral': 1, 'positive': 2}
label_lookup = {f"LABEL_{v}": k for k, v in labels_dict.items()}

# --- 3. Test with new tweets! ---
tweet1 = "Yeh phone service bahut bekar hai. Not good at all."
tweet2 = "Wow, kya fast delivery hai! I am very happy."
tweet3 = "Price theek hai, average product."



for tweet in [tweet1, tweet2, tweet3]:
    result = sentiment_pipeline(tweet)[0]

    # Look up the human-readable label
    sentiment = label_lookup[result['label']]

    print(f"\nTweet: '{tweet}'")
    print(f"Sentiment: {sentiment} (Score: {result['score']:.4f})")

Loading model from 'my_final_hinglish_model'...


Device set to use cuda:0


Model loaded successfully.

Tweet: 'Yeh phone service bahut bekar hai. Not good at all.'
Sentiment: neutral (Score: 0.7840)

Tweet: 'Wow, kya fast delivery hai! I am very happy.'
Sentiment: positive (Score: 0.6939)

Tweet: 'Price theek hai, average product.'
Sentiment: neutral (Score: 0.8904)


In [ ]:
import numpy as np
import evaluate

print("--- Running Evaluation on Test Set ---")

# --- 1. Get predictions from the trainer ---
# This runs your model over the entire hidden test set.
# 'tokenized_dataset' should still be in memory from Step 3
predictions = trainer.predict(tokenized_dataset["test"])



# --- 2. Get the actual numbers ---
# 'predictions.predictions' are the raw outputs (logits)
# We use argmax to get the highest-scoring prediction (0, 1, or 2)
y_preds = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids # These are the true, correct labels

# --- 3. Load the metrics ---
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

# --- 4. Calculate and print the results ---
acc = acc_metric.compute(predictions=y_preds, references=y_true)
f1 = f1_metric.compute(predictions=y_preds, references=y_true, average="weighted")

print(f"\n--- Model Performance ---")
print(f"Accuracy: {acc['accuracy'] * 100:.2f}%")
print(f"F1-Score (Weighted): {f1['f1']:.4f}")

--- Running Evaluation on Test Set ---



--- Model Performance ---
Accuracy: 66.49%
F1-Score (Weighted): 0.6636


In [ ]:
# --- 1. Zip the model folder ---
print("Zipping the model folder...")
!zip -r my_final_hinglish_model.zip my_final_hinglish_model
print("Zipping complete.")



# --- 2. Download the zip file ---
from google.colab import files
print("Downloading 'my_final_hinglish_model.zip'. Please wait...")
files.download('my_final_hinglish_model.zip')

Zipping the model folder...
updating: my_final_hinglish_model/ (stored 0%)
updating: my_final_hinglish_model/training_args.bin (deflated 54%)
updating: my_final_hinglish_model/special_tokens_map.json (deflated 52%)
updating: my_final_hinglish_model/sentencepiece.bpe.model (deflated 49%)
updating: my_final_hinglish_model/model.safetensors


zip error: Interrupted (aborting)
Zipping complete.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("Drive mounted successfully.")

Mounting Google Drive...
Mounted at /content/drive
Drive mounted successfully.


In [ ]:
print("Copying zip file to your Google Drive...")
!cp my_final_hinglish_model.zip /content/drive/MyDrive/
print("File copied to your Google Drive!")

Copying zip file to your Google Drive...
File copied to your Google Drive!


In [ ]:
!pip install --upgrade transformers datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 24.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.


In [ ]:
import transformers
print(f"Transformers version: {transformers.__version__}")

Transformers version: 4.57.1


In [ ]:
from google.colab import drive
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("Drive mounted successfully.")

Mounting Google Drive...
Mounted at /content/drive
Drive mounted successfully.


In [ ]:
print("Unzipping your first model (v1) from Google Drive...")
# Note: We are using the *original* zip file name
!unzip /content/drive/MyDrive/my_final_hinglish_model.zip
print("Your 66% model is now unzipped!")

Unzipping your first model (v1) from Google Drive...
Archive:  /content/drive/MyDrive/my_final_hinglish_model.zip
   creating: my_final_hinglish_model/
  inflating: my_final_hinglish_model/training_args.bin  
  inflating: my_final_hinglish_model/special_tokens_map.json  
  inflating: my_final_hinglish_model/sentencepiece.bpe.model  
  inflating: my_final_hinglish_model/model.safetensors  
  inflating: my_final_hinglish_model/config.json  
  inflating: my_final_hinglish_model/tokenizer.json  
  inflating: my_final_hinglish_model/tokenizer_config.json  
Your 66% model is now unzipped!


In [ ]:
from transformers import pipeline

# Load the model from the folder we just unzipped
sentiment_pipeline_v1 = pipeline(
    "text-classification",
    model="my_final_hinglish_model",
    tokenizer="my_final_hinglish_model"
)

# Create a lookup for the labels
labels_dict = {'negative': 0, 'neutral': 1, 'positive': 2}
label_lookup = {f"LABEL_{v}": k for k, v in labels_dict.items()}

# Get a prediction
tweet = "kya mast movie hai bhai! maza aa gaya"
result = sentiment_pipeline_v1(tweet)[0]
sentiment = label_lookup[result['label']]

print("--- Testing your 66% model (v1) ---")
print(f"Tweet: '{tweet}'")
print(f"Sentiment: {sentiment} (Score: {result['score']:.4f})")

Device set to use cpu


--- Testing your 66% model (v1) ---
Tweet: 'kya mast movie hai bhai! maza aa gaya'
Sentiment: positive (Score: 0.7557)
